# To-do:

- Relevante Jahre ausgewaehlt --> 1999 bis 2015
  - Leere Werte in den Jahre befuellen mit Imputor oder alternative.
- Join durchfuehren der Tabellen, damit in einer großen Tabelle alle relevanten Spalten inkl. Beschreibungen zu finden sind.
- Interessante Indikatoren ausgewaehlt.
  - Falls notwendig Indikatoren gruppiern
    - Population, gpd, Education etc.
  - One-hot Encoding fuer Indikatoren -> Falls Zeitreihenanalyse
- Laenderauswahl:
  - Auswahl nach Einwohner, Große, GDP ?
- Spaltenauswahl:
  - Welche Spalten sind noch relevant bis Country, Code, Year? Oder nur bei den 3 bleiben
- Many Plots


Education Outcomes and Inequality: A Global Analysis of Completion Rates, Education Spending, and Gender Gaps


### 0. Imports and Setup


In [ ]:
#pip list --format=freeze > requirements.txt

In [ ]:
import zipfile

with zipfile.ZipFile("./data/EdStats_CSV.zip", 'r') as zip:
    zip.extractall("./data")

In [48]:
import pandas as pd
import numpy as np
import sklearn
import seaborn as sns
import scipy
import plotly
import statsmodels
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
np.random.seed(SEED)

DATE_RANGE_INDEX = range(1999,2016)
MIN_GDP_RATIO_TO_GERMANY = 0.1

In [2]:
df_data = pd.read_csv("./data/EdStatsData.csv")

df_country_series = pd.read_csv("./data/EdStatsCountry-Series.csv")
df_country = pd.read_csv("./data/EdStatsCountry.csv")

df_footnote= pd.read_csv("./data/EdStatsFootNote.csv")
df_series = pd.read_csv("./data/EdStatsSeries.csv")

In [3]:
def drop_columns(df, columns_to_drop):
    dropped_cols = [col for col in df.columns if col not in columns_to_drop]
    if dropped_cols:
        print("Dropped:", dropped_cols)
        df = df.drop(columns=dropped_cols)
    return df

In [ ]:
display(df_data.info())
display(df_country.info())
display(df_series.info())

In [ ]:
df_country.sample(10)

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data,Unnamed: 31
24,BLZ,Belize,Belize,Belize,BZ,Belize dollar,NaN,Latin America & Caribbean,Upper middle income,BZ,...,General Data Dissemination System (GDDS),2010,"Multiple Indicator Cluster Survey (MICS), 2011","Expenditure survey/budget survey (ES/BS), 2011",NaN,NaN,NaN,2012.0,2000,NaN
6,ARE,United Arab Emirates,United Arab Emirates,United Arab Emirates,AE,U.A.E. dirham,April 2013 database update: Based on data from...,Middle East & North Africa,High income: nonOECD,AE,...,General Data Dissemination System (GDDS),2010,NaN,NaN,NaN,2012,NaN,2011.0,2005,NaN
222,TZA,Tanzania,Tanzania,United Republic of Tanzania,TZ,Tanzanian shilling,NaN,Sub-Saharan Africa,Low income,TZ,...,General Data Dissemination System (GDDS),2012,HIV/Maternal and Child Health (HIV/MCH) Servic...,"Expenditure survey/budget survey (ES/BS), 2011/12",NaN,2007/08,2007.0,2012.0,2002,NaN
208,SYC,Seychelles,Seychelles,Republic of Seychelles,SC,Seychelles rupee,April 2013 database update: Based on official ...,Sub-Saharan Africa,Upper middle income,SC,...,General Data Dissemination System (GDDS),2010,NaN,"Budget survey (BS), 2006/07",Yes,2011,NaN,2008.0,2005,NaN
236,XKX,Kosovo,Kosovo,Republic of Kosovo,NaN,Euro,"Kosovo became a World Bank member on June 29, ...",Europe & Central Asia,Lower middle income,KV,...,General Data Dissemination System (GDDS),2011,NaN,"Integrated household survey (IHS), 2011",NaN,NaN,NaN,NaN,NaN,NaN
173,PHL,Philippines,Philippines,Republic of the Philippines,PH,Philippine peso,Source for GNI and net income from abroad is c...,East Asia & Pacific,Lower middle income,PH,...,Special Data Dissemination Standard (SDDS),2010,"Demographic and Health Survey (DHS), 2013","Expenditure survey/budget survey (ES/BS), 2012",Yes,2012,2008.0,2012.0,2009,NaN
197,SRB,Serbia,Serbia,Republic of Serbia,RS,New Serbian dinar,Montenegro declared independence from Serbia a...,Europe & Central Asia,Upper middle income,YF,...,General Data Dissemination System (GDDS),2011,"Multiple Indicator Cluster Survey (MICS), 2010","Integrated household survey (IHS), 2010",Yes,2012,2010.0,NaN,2009,NaN
220,TUR,Turkey,Turkey,Republic of Turkey,TR,New Turkish lira,NaN,Europe & Central Asia,Upper middle income,TR,...,Special Data Dissemination Standard (SDDS),2011,"Demographic and Health Survey (DHS), 2003; Wor...","Labor force survey (LFS), 2009",NaN,NaN,2009.0,2012.0,2003,NaN
9,ASM,American Samoa,American Samoa,American Samoa,AS,U.S. dollar,NaN,East Asia & Pacific,Upper middle income,AS,...,NaN,2010,NaN,NaN,Yes,2007,NaN,NaN,NaN,NaN
113,KNA,St. Kitts and Nevis,St. Kitts and Nevis,St. Kitts and Nevis,KN,East Caribbean dollar,April 2012 database update: Based on official ...,Latin America & Caribbean,High income: nonOECD,KN,...,General Data Dissemination System (GDDS),2011,NaN,NaN,Yes,NaN,NaN,2011.0,NaN,NaN


In [4]:
columns_to_keep_series = ["Indicator Code", "Topic", "Short definition", "Long definition", "Other notes", "Source", "Aggregation method", "Periodicity"]
columns_to_keep_country = ["Country Code", "Currency Unit", "Special Notes", "Region", "Income Group", "System of trade"]

In [5]:
df_series.rename(columns={"Series Code": "Indicator Code"}, inplace=True)

df_series = drop_columns(df_series, columns_to_keep_series)
df_country = drop_columns(df_country, columns_to_keep_country)
display(df_series.info())
display(df_country.info())

Dropped: ['Indicator Name', 'Unit of measure', 'Base Period', 'Limitations and exceptions', 'Notes from original source', 'General comments', 'Statistical concept and methodology', 'Development relevance', 'Related source links', 'Other web links', 'Related indicators', 'License Type', 'Unnamed: 20']
Dropped: ['Short Name', 'Table Name', 'Long Name', '2-alpha code', 'WB-2 code', 'National accounts base year', 'National accounts reference year', 'SNA price valuation', 'Lending category', 'Other groups', 'System of National Accounts', 'Alternative conversion factor', 'PPP survey year', 'Balance of Payments Manual in use', 'External debt Reporting status', 'Government Accounting concept', 'IMF data dissemination standard', 'Latest population census', 'Latest household survey', 'Source of most recent Income and expenditure data', 'Vital registration complete', 'Latest agricultural census', 'Latest industrial data', 'Latest trade data', 'Latest water withdrawal data', 'Unnamed: 31']
<class 

None

<class 'pandas.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Country Code     241 non-null    str  
 1   Currency Unit    215 non-null    str  
 2   Special Notes    145 non-null    str  
 3   Region           214 non-null    str  
 4   Income Group     214 non-null    str  
 5   System of trade  200 non-null    str  
dtypes: str(6)
memory usage: 11.4 KB


None

In [6]:
# Only the columns from 1999 to 2015 seem well documented. 

df_data.drop("Unnamed: 69", axis=1, inplace=True)

cols_to_drop = [
    col for col in df_data.select_dtypes(exclude="O").columns
    if isinstance(col, str) and (int(col) < min(DATE_RANGE_INDEX) or int(col) > max(DATE_RANGE_INDEX))
        
]
df_data.drop(cols_to_drop, axis=1, inplace=True)

In [7]:
# Country Code und Indicator Code / Series Code zum mergen verwenden -> SQL join
display(df_data.info())
#display(df_country.info())

df_combined = df_data.merge(df_country, on="Country Code", how="left")
df_combined = df_combined.merge(df_series, on="Indicator Code", how="left")


display(df_combined.sample(10))
df_combined.columns

<class 'pandas.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country Name    886930 non-null  str    
 1   Country Code    886930 non-null  str    
 2   Indicator Name  886930 non-null  str    
 3   Indicator Code  886930 non-null  str    
 4   1999            118839 non-null  float64
 5   2000            176676 non-null  float64
 6   2001            123509 non-null  float64
 7   2002            124205 non-null  float64
 8   2003            130363 non-null  float64
 9   2004            128814 non-null  float64
 10  2005            184108 non-null  float64
 11  2006            140312 non-null  float64
 12  2007            137272 non-null  float64
 13  2008            134387 non-null  float64
 14  2009            142108 non-null  float64
 15  2010            242442 non-null  float64
 16  2011            146012 non-null  float64
 17  2012            14726

None

,Country Name,Country Code,Indicator Name,Indicator Code,1999,2000,2001,2002,2003,2004,...,Region,Income Group,System of trade,Topic,Short definition,Long definition,Periodicity,Other notes,Aggregation method,Source
471498,"Korea, Rep.",KOR,"Population, age 2, male",SP.POP.AG02.MA.UN,334120.0,326280.0,314940.0,300635.0,284941.0,270063.0,...,East Asia & Pacific,High income: OECD,General trade system,Population,"Population, age 2, male refers to the male pop...","Population, age 2, male refers to the male pop...",NaN,NaN,NaN,UNESCO Institute for Statistics
589164,Myanmar,MMR,SABER: (School Finance) Policy Goal 4 Lever 2:...,SABER.SCH.FNNC.GOAL4.LVL2,NaN,NaN,NaN,NaN,NaN,NaN,...,East Asia & Pacific,Low income,General trade system,School Finance (SABER),Data Interpretation: 1=Latent; 2=Emerging; 3=E...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,NaN,NaN,NaN,Systems Approach for Better Education Results ...
25448,European Union,EUU,Wittgenstein Projection: Percentage of the pop...,PRJ.ATT.2039.3.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Attainment,NaN,Share of the population of the stated age grou...,NaN,NaN,NaN,Wittgenstein Centre for Demography and Global ...
667853,Puerto Rico,PRI,EGRA: Correct Non-Words Read Per Minute (Mean)...,LO.EGRA.NCWPM.ICI.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,Latin America & Caribbean,High income: nonOECD,General trade system,Learning Outcomes,Average total number of invented/nonsense word...,Average total number of invented/nonsense word...,NaN,EGRA,NaN,Early Grade Reading Assessment (EGRA): https:/...
53207,Lower middle income,LMC,Percentage of graduates from Agriculture progr...,SE.TER.GRAD.FE.AG.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Tertiary,NaN,Number of female graduates in agriculture expr...,NaN,NaN,NaN,UNESCO Institute for Statistics
356247,Germany,DEU,EGRA: Correct Letter Names Read Per Minute (Me...,LO.EGRA.CLPM.AMH.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,Europe & Central Asia,High income: OECD,Special trade system,Learning Outcomes,Average number of letter names that students c...,Average number of letter names that students c...,NaN,EGRA,NaN,Early Grade Reading Assessment (EGRA): https:/...
567157,Moldova,MDA,SABER: (School Autonomy Accountability) Policy...,SABER.SCH.ATNM.GOAL4,NaN,NaN,NaN,NaN,NaN,NaN,...,Europe & Central Asia,Lower middle income,General trade system,School Autonomy and Accountability (SABER),Data Interpretation: 1=Latent; 2=Emerging; 3=E...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,NaN,NaN,NaN,Systems Approach for Better Education Results ...
321958,Estonia,EST,"UIS: Adjusted net attendance rate, primary, fo...",UIS.NARA.1.Q4.GPI,NaN,NaN,NaN,NaN,NaN,NaN,...,Europe & Central Asia,High income: OECD,Special trade system,Education Equality,Ratio of female adjusted net attendance rate t...,Ratio of female adjusted net attendance rate t...,NaN,NaN,NaN,UNESCO Institute for Statistics
755878,St. Lucia,LCA,EGRA: Listening Comprehension - Share of stude...,LO.EGRA.LSTN.0.ICI.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,Latin America & Caribbean,Upper middle income,Special trade system,Learning Outcomes,Share of students scoring zero percent on the ...,Share of students scoring zero percent on the ...,NaN,EGRA,NaN,Early Grade Reading Assessment (EGRA): https:/...
803262,Togo,TGO,DHS: Secondary completion rate. Rural,HH.DHS.SCR.R,NaN,NaN,NaN,NaN,NaN,NaN,...,Sub-Saharan Africa,Low income,Special trade system,Education Equality,Secondary completion rate. Rural is the total ...,Secondary completion rate. Rural is the total ...,NaN,NaN,NaN,Demographic and Health Surveys (DHS)


Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007',
       '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015',
       'Currency Unit', 'Special Notes', 'Region', 'Income Group',
       'System of trade', 'Topic', 'Short definition', 'Long definition',
       'Periodicity', 'Other notes', 'Aggregation method', 'Source'],
      dtype='str')

In [8]:
# Rearange the columns, so that the nummeric values are after the categorical
df_combined_cat = df_combined.select_dtypes(include="O")
df_combined_num = df_combined.select_dtypes(exclude="O")

df_combined = pd.concat([df_combined_cat, df_combined_num], axis=1)

C:\Users\alexe\AppData\Local\Temp\ipykernel_25728\2334459849.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_combined_cat = df_combined.select_dtypes(include="O")


In [ ]:
df_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Data columns (total 33 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Country Name        886930 non-null  str    
 1   Country Code        886930 non-null  str    
 2   Indicator Name      886930 non-null  str    
 3   Indicator Code      886930 non-null  str    
 4   Currency Unit       787975 non-null  str    
 5   Special Notes       531425 non-null  str    
 6   Region              784310 non-null  str    
 7   Income Group        784310 non-null  str    
 8   System of trade     733000 non-null  str    
 9   Topic               874104 non-null  str    
 10  Short definition    520784 non-null  str    
 11  Long definition     874104 non-null  str    
 12  Periodicity         23958 non-null   str    
 13  Other notes         132616 non-null  str    
 14  Aggregation method  11374 non-null   str    
 15  Source              874104 non-null  str    


In [ ]:
df_combined.sample(10)

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
316766,Eritrea,ERI,MICS: Primary completion rate. Quintile 2,HH.MICS.PCR.Q2,Eritrean nakfa,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Low income,NaN,Education Equality,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
446810,Jamaica,JAM,Under-age enrolment ratio in secondary educati...,UIS.UAPP.23.M,Jamaican dollar,April 2013 database update: Based on official ...,Latin America & Caribbean,Upper middle income,General trade system,Secondary,...,NaN,NaN,NaN,NaN,NaN,0.34384,NaN,0.26515,0.28594,NaN
47010,Low & middle income,LMY,TIMSS: Female 4th grade students reaching the ...,LO.TIMSS.MAT4.INT.FE,NaN,Low and middle income group aggregate (all dev...,NaN,NaN,NaN,Learning Outcomes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
504639,Libya,LBY,"Prevalence of HIV, total (% of population ages...",SH.DYN.AIDS.ZS,Libyan dinar,National accounts data are revised from 2010 t...,Middle East & North Africa,Upper middle income,General trade system,Health: Risk factors,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142486,"Bahamas, The",BHS,UIS: Percentage of population age 25+ with at ...,UIS.EA.7T8.AG25T99.GPI,Bahamian dollar,NaN,Latin America & Caribbean,High income: nonOECD,General trade system,Attainment,...,NaN,NaN,NaN,NaN,1.2,NaN,NaN,NaN,NaN,NaN
672254,Qatar,QAT,MICS: Net attendance rate. Primary. Quintile 5,HH.MICS.NAR.1.Q5,Qatari riyal,NaN,Middle East & North Africa,High income: nonOECD,Special trade system,Education Equality,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
592148,Namibia,NAM,Percentage of teachers in upper secondary educ...,UIS.TRTP.3.GPI,Namibian dollar,Fiscal year end: March 31; reporting period fo...,Sub-Saharan Africa,Upper middle income,General trade system,Teachers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47293,Low & middle income,LMY,"UIS: Total net attendance rate, lower secondar...",UIS.NART.2.Q4.M,NaN,Low and middle income group aggregate (all dev...,NaN,NaN,NaN,Education Equality,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
662872,Poland,POL,"UIS: Net attendance rate, primary, gender pari...",UIS.NAR.1.GPI,Polish zloty,NaN,Europe & Central Asia,High income: OECD,Special trade system,Education Equality,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
738813,South Africa,ZAF,PIAAC: Male adults by numeracy proficiency lev...,LO.PIAAC.NUM.MA.3,South African rand,Fiscal year end: March 31; reporting period fo...,Sub-Saharan Africa,Upper middle income,General trade system,Learning Outcomes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# df_series.rename(columns={"Series Code": "Indicator Code"}, inplace=True)

# size = len(df_series.columns)
# i = 0
# count = 0
# while count <= size -1:
#     count += 1
#     #print(df_series.columns[i], i , "")
#     if df_series.columns[i] not in columns_to_keep_series:
#         #print("Dropped:",df_series.columns[i])
#         df_series.drop(df_series.columns[i], axis=1, inplace=True)
#         i-=1
#     i+=1

In [ ]:
display(df_data.value_counts("Indicator Code"))
df_footnote.value_counts("SeriesCode")
df_country_series.value_counts("SeriesCode")

SeriesCode
SP.POP.TOTL          211
SP.POP.GROW          211
NY.GDP.PCAP.PP.CD     19
NY.GDP.PCAP.PP.KD     19
NY.GNP.PCAP.PP.CD     19
NY.GDP.MKTP.PP.KD     14
NY.GNP.MKTP.PP.CD     14
NY.GDP.MKTP.PP.CD     14
SP.POP.1564.TO.ZS     13
SP.POP.TOTL.MA.ZS     13
SP.POP.TOTL.FE.ZS     13
SP.POP.0014.TO.ZS     13
NY.GNP.PCAP.CD         6
NY.GDP.PCAP.CD         5
NY.GDP.PCAP.KD         5
SP.POP.1564.MA.IN      4
SP.POP.0014.TO         4
SP.POP.1564.TO         4
SP.POP.1564.FE.IN      4
SP.POP.0014.MA.IN      4
SP.POP.0014.FE.IN      4
Name: count, dtype: int64

In [ ]:
display(df_data, df_footnote, df_country_series)
# df_series -> Enthaelt Beschreibung bezueglich des Indikators. Also gut zum nachschlagen

,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
0,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.F,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.GPI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.M,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arab World,ARB,"Adjusted net enrolment rate, primary, both sex...",SE.PRM.TENR,54.822121,54.894138,56.209438,57.267109,57.991138,59.36554,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886925,Zimbabwe,ZWE,"Youth illiterate population, 15-24 years, male...",UIS.LP.AG15T24.M,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886926,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, b...",SE.ADT.1524.LT.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886927,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, f...",SE.ADT.1524.LT.FE.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886928,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, g...",SE.ADT.1524.LT.FM.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,CountryCode,SeriesCode,Year,DESCRIPTION,Unnamed: 4
0,ABW,SE.PRE.ENRL.FE,YR2001,Country estimation.,NaN
1,ABW,SE.TER.TCHR.FE,YR2005,Country estimation.,NaN
2,ABW,SE.PRE.TCHR.FE,YR2000,Country estimation.,NaN
3,ABW,SE.SEC.ENRL.GC,YR2004,Country estimation.,NaN
4,ABW,SE.PRE.TCHR,YR2006,Country estimation.,NaN
...,...,...,...,...,...
643633,ZWE,SH.DYN.MORT,YR2007,Uncertainty bound is 91.6 - 109.3,NaN
643634,ZWE,SH.DYN.MORT,YR2014,Uncertainty bound is 54.3 - 76,NaN
643635,ZWE,SH.DYN.MORT,YR2015,Uncertainty bound is 48.3 - 73.3,NaN
643636,ZWE,SH.DYN.MORT,YR2017,5-year average value between 0s and 5s,NaN


,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
0,ABW,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
1,ABW,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
2,AFG,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
3,AFG,NY.GDP.PCAP.PP.CD,Estimates are based on regression.,NaN
4,AFG,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
...,...,...,...,...
608,ZAF,SP.POP.GROW,"Data sources : Statistics South Africa, United...",NaN
609,ZMB,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
610,ZMB,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
611,ZWE,SP.POP.TOTL,Data sources : United Nations World Population...,NaN


,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
0,BAR.NOED.1519.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15-19 with...,Percentage of female population age 15-19 with...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BAR.NOED.1519.ZS,Attainment,Barro-Lee: Percentage of population age 15-19 ...,Percentage of population age 15-19 with no edu...,Percentage of population age 15-19 with no edu...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BAR.NOED.15UP.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15+ with n...,Percentage of female population age 15+ with n...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BAR.NOED.15UP.ZS,Attainment,Barro-Lee: Percentage of population age 15+ wi...,Percentage of population age 15+ with no educa...,Percentage of population age 15+ with no educa...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BAR.NOED.2024.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 20-24 with...,Percentage of female population age 20-24 with...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3660,UIS.XUNIT.USCONST.3.FSGOV,Expenditures,Government expenditure per upper secondary stu...,NaN,"Average total (current, capital and transfers)...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3661,UIS.XUNIT.USCONST.4.FSGOV,Expenditures,Government expenditure per post-secondary non-...,NaN,"Average total (current, capital and transfers)...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3662,UIS.XUNIT.USCONST.56.FSGOV,Expenditures,Government expenditure per tertiary student (c...,NaN,"Average total (current, capital and transfers)...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3663,XGDP.23.FSGOV.FDINSTADM.FFD,Expenditures,Government expenditure in secondary institutio...,"Total general (local, regional and central) go...","Total general (local, regional and central) go...",NaN,NaN,NaN,Secondary,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 0.1. First Look


In [ ]:
df_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country Name    886930 non-null  str    
 1   Country Code    886930 non-null  str    
 2   Indicator Name  886930 non-null  str    
 3   Indicator Code  886930 non-null  str    
 4   1999            118839 non-null  float64
 5   2000            176676 non-null  float64
 6   2001            123509 non-null  float64
 7   2002            124205 non-null  float64
 8   2003            130363 non-null  float64
 9   2004            128814 non-null  float64
 10  2005            184108 non-null  float64
 11  2006            140312 non-null  float64
 12  2007            137272 non-null  float64
 13  2008            134387 non-null  float64
 14  2009            142108 non-null  float64
 15  2010            242442 non-null  float64
 16  2011            146012 non-null  float64
 17  2012            14726

In [ ]:
display(df_data["Indicator Code"].value_counts()) # 10*242*366+5*242 = 886930
display(df_data["Country Code"].value_counts())
display(df_data["Country Name"].value_counts())

Indicator Code
UIS.NERA.2              242
UIS.NERA.2.F            242
UIS.NERA.2.GPI          242
UIS.NERA.2.M            242
SE.PRM.TENR             242
                       ... 
UIS.LP.AG15T24.M        242
SE.ADT.1524.LT.ZS       242
SE.ADT.1524.LT.FE.ZS    242
SE.ADT.1524.LT.FM.ZS    242
SE.ADT.1524.LT.MA.ZS    242
Name: count, Length: 3665, dtype: int64

Country Code
ARB    3665
EAS    3665
EAP    3665
EMU    3665
ECS    3665
       ... 
VIR    3665
PSE    3665
YEM    3665
ZMB    3665
ZWE    3665
Name: count, Length: 242, dtype: int64

Country Name
Arab World                                     3665
East Asia & Pacific                            3665
East Asia & Pacific (excluding high income)    3665
Euro area                                      3665
Europe & Central Asia                          3665
                                               ... 
Virgin Islands (U.S.)                          3665
West Bank and Gaza                             3665
Yemen, Rep.                                    3665
Zambia                                         3665
Zimbabwe                                       3665
Name: count, Length: 242, dtype: int64

In [ ]:
df_footnote["SeriesCode"].value_counts()

SeriesCode
SH.DYN.MORT             9226
SE.PRM.AGES             8771
SE.PRM.DURS             8771
SE.SEC.DURS             8619
SE.SEC.AGES             8581
                        ... 
LO.TIMSS.MAT4.HI.MA        1
LO.TIMSS.MAT4.INT.FE       1
LO.TIMSS.MAT4.INT.MA       1
LO.TIMSS.MAT4.LOW.FE       1
LO.TIMSS.MAT4.BL.FE        1
Name: count, Length: 1558, dtype: int64

In [ ]:
df_series.sample(10)

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
1960,PRJ.POP.2529.1.MF,Attainment,Projection: Population age 25-29 in thousands ...,NaN,Total population in thousands in the specified...,NaN,NaN,Projections (2010 to 2100),NaN,NaN,...,NaN,NaN,Wittgenstein Centre for Demography and Global ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3186,UIS.NART.2.Q2.GPI,Education Equality,"UIS: Total net attendance rate, lower secondar...",Ratio of female total net attendance rate to m...,Ratio of female total net attendance rate to m...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2813,UIS.E.2.GPV.GUK.F,Secondary,Enrolment in lower secondary general education...,NaN,Total number of female students enrolled in lo...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1104,LO.LLECE.REA6.2.FE,Learning Outcomes,LLECE: Female 6th grade students by reading pr...,Percentage of 6th grade female students scorin...,Percentage of 6th grade female students scorin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Latin American Laboratory for Assessment of th...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2942,UIS.FEP.56.FUK,Tertiary,Percentage of students enrolled in programmes ...,NaN,Total number of female students in unspecified...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
321,BAR.TER.ICMP.5559.ZS,Attainment,Barro-Lee: Percentage of population age 55-59 ...,Percentage of population age 55-59 with tertia...,Percentage of population age 55-59 with tertia...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1754,PRJ.ATT.2039.NED.MF,Attainment,Projection: Percentage of the population age 2...,NaN,Share of the population of the stated age grou...,NaN,NaN,Projections (2010 to 2100),NaN,NaN,...,NaN,NaN,Wittgenstein Centre for Demography and Global ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1462,LO.PISA.REA.MA,Learning Outcomes,PISA: Mean performance on the reading scale. Male,Average score of 15-year-old male students on ...,Average score of 15-year-old male students on ...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,OECD Programme for International Student Asses...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2470,SP.POP.1117.TO.UN,Population,"Population, ages 11-17, total","Population, ages 11-17, total is the total pop...","Population, ages 11-17, total is the total pop...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics (Derived),NaN,NaN,NaN,NaN,NaN,NaN,NaN
3332,UIS.REPP.1.G4.M,Primary,Percentage of repeaters in Grade 4 of primary ...,NaN,Total number of male pupils in the specified g...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Kurze Regelungen:
# - Nur Nominal und Ordinale Skalen werden beibehalten. -> Beschreibungen, Redundante Namen, Lizensen werden verworfen.
# - Nur Zeitachse von 1999 bis 2026 wird behalten
# - 
display("df_data:", df_data.columns) # Zeitliche Angaben von 1970 bis 2100 + Spalte 69. -> Als Relevant definier ich 2000 bis 2026, Rest wird verworfen.

display("df_country_series:", df_country_series.columns)
display("df_country:",df_country.columns)

display("df_footnote",df_footnote.columns)
display("df_series:",df_series.columns)

'df_data:'

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007',
       '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015'],
      dtype='str')

'df_country_series:'

Index(['CountryCode', 'SeriesCode', 'DESCRIPTION', 'Unnamed: 3'], dtype='str')

'df_country:'

Index(['Country Code', 'Short Name', 'Table Name', 'Long Name', '2-alpha code',
       'Currency Unit', 'Special Notes', 'Region', 'Income Group', 'WB-2 code',
       'National accounts base year', 'National accounts reference year',
       'SNA price valuation', 'Lending category', 'Other groups',
       'System of National Accounts', 'Alternative conversion factor',
       'PPP survey year', 'Balance of Payments Manual in use',
       'External debt Reporting status', 'System of trade',
       'Government Accounting concept', 'IMF data dissemination standard',
       'Latest population census', 'Latest household survey',
       'Source of most recent Income and expenditure data',
       'Vital registration complete', 'Latest agricultural census',
       'Latest industrial data', 'Latest trade data',
       'Latest water withdrawal data', 'Unnamed: 31'],
      dtype='str')

'df_footnote'

Index(['CountryCode', 'SeriesCode', 'Year', 'DESCRIPTION', 'Unnamed: 4'], dtype='str')

'df_series:'

Index(['Series Code', 'Topic', 'Indicator Name', 'Short definition',
       'Long definition', 'Unit of measure', 'Periodicity', 'Base Period',
       'Other notes', 'Aggregation method', 'Limitations and exceptions',
       'Notes from original source', 'General comments', 'Source',
       'Statistical concept and methodology', 'Development relevance',
       'Related source links', 'Other web links', 'Related indicators',
       'License Type', 'Unnamed: 20'],
      dtype='str')

In [ ]:
display(df_data.columns)
display(df_data.sample(10))
display(df_data.info())

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978',
       '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987',
       '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996',
       '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2020', '2025', '2030', '2035', '2040', '2045',
       '2050', '2055', '2060', '2065', '2070', '2075', '2080', '2085', '2090',
       '2095', '2100', 'Unnamed: 69'],
      dtype='str')

,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
471498,"Korea, Rep.",KOR,"Population, age 2, male",SP.POP.AG02.MA.UN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
589164,Myanmar,MMR,SABER: (School Finance) Policy Goal 4 Lever 2:...,SABER.SCH.FNNC.GOAL4.LVL2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25448,European Union,EUU,Wittgenstein Projection: Percentage of the pop...,PRJ.ATT.2039.3.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
667853,Puerto Rico,PRI,EGRA: Correct Non-Words Read Per Minute (Mean)...,LO.EGRA.NCWPM.ICI.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53207,Lower middle income,LMC,Percentage of graduates from Agriculture progr...,SE.TER.GRAD.FE.AG.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
356247,Germany,DEU,EGRA: Correct Letter Names Read Per Minute (Me...,LO.EGRA.CLPM.AMH.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
567157,Moldova,MDA,SABER: (School Autonomy Accountability) Policy...,SABER.SCH.ATNM.GOAL4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
321958,Estonia,EST,"UIS: Adjusted net attendance rate, primary, fo...",UIS.NARA.1.Q4.GPI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
755878,St. Lucia,LCA,EGRA: Listening Comprehension - Share of stude...,LO.EGRA.LSTN.0.ICI.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
803262,Togo,TGO,DHS: Secondary completion rate. Rural,HH.DHS.SCR.R,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Data columns (total 70 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country Name    886930 non-null  str    
 1   Country Code    886930 non-null  str    
 2   Indicator Name  886930 non-null  str    
 3   Indicator Code  886930 non-null  str    
 4   1970            72288 non-null   float64
 5   1971            35537 non-null   float64
 6   1972            35619 non-null   float64
 7   1973            35545 non-null   float64
 8   1974            35730 non-null   float64
 9   1975            87306 non-null   float64
 10  1976            37483 non-null   float64
 11  1977            37574 non-null   float64
 12  1978            37576 non-null   float64
 13  1979            36809 non-null   float64
 14  1980            89122 non-null   float64
 15  1981            38777 non-null   float64
 16  1982            37511 non-null   float64
 17  1983            38460

None

In [ ]:
display(df_footnote.columns)
display(df_footnote.sample(10))
display(df_footnote.info())

Index(['CountryCode', 'SeriesCode', 'Year', 'DESCRIPTION', 'Unnamed: 4'], dtype='str')

,CountryCode,SeriesCode,Year,DESCRIPTION,Unnamed: 4
101137,CMR,SE.PRM.TCHR.FE.ZS,YR2009,Estimated,NaN
627145,VUT,UIS.DR.2.GPV.G1.F,YR2003,UNESCO Institute for Statistics (UIS) estimate,NaN
599250,TZA,SE.SEC.ENRL,YR1986,Country Data,NaN
292808,KGZ,SE.PRM.CMPL.ZS,YR2011,Estimated,NaN
641981,ZMB,SE.TER.ENRR.FE,YR1982,Country Data,NaN
619565,VCT,UIS.T.23.V.F,YR2004,UNESCO Institute for Statistics (UIS) estimate,NaN
595173,TUV,SP.PRE.TOTL.FE.IN,YR2008,National Estimation,NaN
633839,WLD,UIS.SLE.56.GPI,YR2014,UNESCO Institute for Statistics (UIS) estimate,NaN
231615,HIC,SE.PRM.TCHR,YR1983,UIS Estimation,NaN
437411,MYS,SE.SCH.LIFE,YR1992,Country Data,NaN


<class 'pandas.DataFrame'>
RangeIndex: 643638 entries, 0 to 643637
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CountryCode  643638 non-null  str    
 1   SeriesCode   643638 non-null  str    
 2   Year         643638 non-null  str    
 3   DESCRIPTION  643638 non-null  str    
 4   Unnamed: 4   0 non-null       float64
dtypes: float64(1), str(4)
memory usage: 24.6 MB


None

In [ ]:
display(df_country.columns)
display(df_country.sample(10))
display(df_country.info())
display(df_country.describe())

Index(['Country Code', 'Short Name', 'Table Name', 'Long Name', '2-alpha code',
       'Currency Unit', 'Special Notes', 'Region', 'Income Group', 'WB-2 code',
       'National accounts base year', 'National accounts reference year',
       'SNA price valuation', 'Lending category', 'Other groups',
       'System of National Accounts', 'Alternative conversion factor',
       'PPP survey year', 'Balance of Payments Manual in use',
       'External debt Reporting status', 'System of trade',
       'Government Accounting concept', 'IMF data dissemination standard',
       'Latest population census', 'Latest household survey',
       'Source of most recent Income and expenditure data',
       'Vital registration complete', 'Latest agricultural census',
       'Latest industrial data', 'Latest trade data',
       'Latest water withdrawal data', 'Unnamed: 31'],
      dtype='str')

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data,Unnamed: 31
67,ETH,Ethiopia,Ethiopia,Federal Democratic Republic of Ethiopia,ET,Ethiopian birr,Fiscal year end: July 7; reporting period for ...,Sub-Saharan Africa,Low income,ET,...,General Data Dissemination System (GDDS),2007,"Demographic and Health Survey (DHS), 2011","Expenditure survey/budget survey (ES/BS), 2010/11",NaN,NaN,2009.0,2012.0,2002,NaN
34,CHE,Switzerland,Switzerland,Switzerland,CH,Swiss franc,NaN,Europe & Central Asia,High income: OECD,CH,...,Special Data Dissemination Standard (SDDS),2010,NaN,"Expenditure survey/budget survey (ES/BS), 2000",Yes,2008,2007.0,2012.0,2000,NaN
172,PER,Peru,Peru,Republic of Peru,PE,Peruvian new sol,NaN,Latin America & Caribbean,Upper middle income,PE,...,Special Data Dissemination Standard (SDDS),2007,Continuous Demographic and Health Survey (DHS)...,"Integrated household survey (IHS), 2012",NaN,2012,2010.0,2012.0,2000,NaN
45,CRI,Costa Rica,Costa Rica,Republic of Costa Rica,CR,Costa Rican colon,NaN,Latin America & Caribbean,Upper middle income,CR,...,Special Data Dissemination Standard (SDDS),2011,"Multiple Indicator Cluster Survey (MICS), 2011","Integrated household survey (IHS), 2012",Yes,2014,2010.0,2012.0,1997,NaN
51,DEU,Germany,Germany,Federal Republic of Germany,DE,Euro,A simple multiplier is used to convert the nat...,Europe & Central Asia,High income: OECD,DE,...,Special Data Dissemination Standard (SDDS),2011,NaN,"Integrated household survey (IHS), 2000",Yes,2010,2009.0,2012.0,2007,NaN
83,GRC,Greece,Greece,Hellenic Republic,GR,Euro,A simple multiplier is used to convert the nat...,Europe & Central Asia,High income: OECD,GR,...,Special Data Dissemination Standard (SDDS),2011,NaN,"Integrated household survey (IHS), 2000",Yes,2009,2007.0,2012.0,2007,NaN
157,NAC,North America,North America,North America,XU,NaN,North America regional aggregate. There are no...,NaN,NaN,XU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
214,TJK,Tajikistan,Tajikistan,Republic of Tajikistan,TJ,Tajik somoni,NaN,Europe & Central Asia,Low income,TJ,...,General Data Dissemination System (GDDS),2010,"Demographic and Health Survey (DHS), 2012",Living Standards Measurement Study Survey (LSM...,NaN,2013,NaN,2000.0,2006,NaN
50,CZE,Czech Republic,Czech Republic,Czech Republic,CZ,Czech koruna,NaN,Europe & Central Asia,High income: OECD,CZ,...,Special Data Dissemination Standard (SDDS),2011,"World Health Survey (WHS), 2003","Income survey (IS), 1996",Yes,2010,2007.0,2012.0,2007,NaN
212,TGO,Togo,Togo,Republic of Togo,TG,CFA franc,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Low income,TG,...,General Data Dissemination System (GDDS),2010,"Demographic and Health Survey (DHS), 2013",Core Welfare Indicator Questionnaire Survey (C...,NaN,2011/2012,NaN,2012.0,2002,NaN


<class 'pandas.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 32 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Country Code                                       241 non-null    str    
 1   Short Name                                         241 non-null    str    
 2   Table Name                                         241 non-null    str    
 3   Long Name                                          241 non-null    str    
 4   2-alpha code                                       238 non-null    str    
 5   Currency Unit                                      215 non-null    str    
 6   Special Notes                                      145 non-null    str    
 7   Region                                             214 non-null    str    
 8   Income Group                                       214 non-null    str    
 9   WB-2 code            

None

,National accounts reference year,Latest industrial data,Latest trade data,Unnamed: 31
count,32.00000,107.000000,185.000000,0.0
mean,2001.53125,2008.102804,2010.994595,NaN
std,5.24856,2.616834,2.569675,NaN
min,1987.00000,2000.000000,1995.000000,NaN
25%,1996.75000,2007.500000,2011.000000,NaN
50%,2002.00000,2009.000000,2012.000000,NaN
75%,2005.00000,2010.000000,2012.000000,NaN
max,2012.00000,2010.000000,2012.000000,NaN


In [ ]:
display(df_series.columns)
display(df_series.sample(10))
display(df_series.info())

attainment = df_series[df_series["Topic"] == "Attainment"]
attainment

# Fazit bisher: Alles unnötig bis Series Code und eventuell Topic

Index(['Series Code', 'Topic', 'Indicator Name', 'Short definition',
       'Long definition', 'Unit of measure', 'Periodicity', 'Base Period',
       'Other notes', 'Aggregation method', 'Limitations and exceptions',
       'Notes from original source', 'General comments', 'Source',
       'Statistical concept and methodology', 'Development relevance',
       'Related source links', 'Other web links', 'Related indicators',
       'License Type', 'Unnamed: 20'],
      dtype='str')

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
701,LO.EGRA.CWPM.ZERO.ARB.3GRD,Learning Outcomes,EGRA: Oral Reading Fluency - Share of students...,Percentage of students who were unable to read...,Percentage of students who were unable to read...,NaN,NaN,NaN,EGRA,NaN,...,NaN,NaN,Early Grade Reading Assessment (EGRA): https:/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3591,UIS.XGOVEXP.IMF.0,Expenditures,Expenditure on pre-primary as % of total gover...,NaN,"Total general (local, regional and central) go...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3514,UIS.SR.1.G4.M,Primary,"Survival rate to Grade 4 of primary education,...",NaN,Percentage of a cohort of male students enroll...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3241,UIS.NIR.1.AgM1.M,Primary,Net intake rate to Grade 1 of primary educatio...,NaN,Number of new male entrants in the first grade...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1020,LO.LLECE.MAT3.1.FE,Learning Outcomes,LLECE: Female 3rd grade students by mathematic...,Percentage of 3rd grade female students scorin...,Percentage of 3rd grade female students scorin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Latin American Laboratory for Assessment of th...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2132,SABER.SCH.FNNC.GOAL5.LVL2,School Finance (SABER),SABER: (School Finance) Policy Goal 5 Lever 2:...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Systems Approach for Better Education Results ...,NaN,NaN,http://saber.worldbank.org/index.cfm,NaN,NaN,NaN,NaN
1974,PRJ.POP.2529.S1.MA,Attainment,Projection: Population age 25-29 in thousands ...,NaN,Total population in thousands in the specified...,NaN,NaN,Projections (2010 to 2100),NaN,NaN,...,NaN,NaN,Wittgenstein Centre for Demography and Global ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
598,HH.MICS.YRS.1519.R,Education Equality,MICS: Average years of schooling by age group....,Average years of schooling by age group. Age 1...,Average years of schooling by age group. Age 1...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,World Bank staff calculations based on Multipl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2607,SP.SEC.TOTL.FE.IN,Population,Population of the official age for secondary e...,NaN,Female population of the age-group theoretical...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
475,HH.DHS.YRS.1519.Q3,Education Equality,DHS: Average years of schooling by age group. ...,Average years of schooling by age group. Age 1...,Average years of schooling by age group. Age 1...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Demographic and Health Surveys (DHS),NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Data columns (total 21 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Series Code                          3665 non-null   str    
 1   Topic                                3665 non-null   str    
 2   Indicator Name                       3665 non-null   str    
 3   Short definition                     2156 non-null   str    
 4   Long definition                      3665 non-null   str    
 5   Unit of measure                      0 non-null      float64
 6   Periodicity                          99 non-null     str    
 7   Base Period                          314 non-null    str    
 8   Other notes                          552 non-null    str    
 9   Aggregation method                   47 non-null     str    
 10  Limitations and exceptions           14 non-null     str    
 11  Notes from original source           0 no

None

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
0,BAR.NOED.1519.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15-19 with...,Percentage of female population age 15-19 with...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BAR.NOED.1519.ZS,Attainment,Barro-Lee: Percentage of population age 15-19 ...,Percentage of population age 15-19 with no edu...,Percentage of population age 15-19 with no edu...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BAR.NOED.15UP.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15+ with n...,Percentage of female population age 15+ with n...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BAR.NOED.15UP.ZS,Attainment,Barro-Lee: Percentage of population age 15+ wi...,Percentage of population age 15+ with no educa...,Percentage of population age 15+ with no educa...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BAR.NOED.2024.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 20-24 with...,Percentage of female population age 20-24 with...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2909,UIS.EA.S1.AG25T99.F,Attainment,UIS: Percentage of population age 25+ with som...,The percentage of female population (age 25 an...,The percentage of female population (age 25 an...,NaN,NaN,NaN,Single Level Attainment/ Not Cumulative,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2910,UIS.EA.S1.AG25T99.M,Attainment,UIS: Percentage of population age 25+ with som...,The percentage of male population (age 25 and ...,The percentage of male population (age 25 and ...,NaN,NaN,NaN,Single Level Attainment/ Not Cumulative,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2911,UIS.EA.UK.AG25T99,Attainment,UIS: Percentage of population age 25+ with unk...,The percentage of the population (age 25 and o...,The percentage of the population (age 25 and o...,NaN,NaN,NaN,Single Level Attainment/ Not Cumulative,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2912,UIS.EA.UK.AG25T99.F,Attainment,UIS: Percentage of population age 25+ with unk...,The percentage of the female population (age 2...,The percentage of the female population (age 2...,NaN,NaN,NaN,Single Level Attainment/ Not Cumulative,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
display(df_country_series.columns)
display(df_country_series.sample(10))
display(df_country_series.info())

# Fazit bisher:

Index(['CountryCode', 'SeriesCode', 'DESCRIPTION', 'Unnamed: 3'], dtype='str')

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
25,ARG,NY.GDP.MKTP.PP.CD,Estimates are based on regression.,NaN
129,CYP,NY.GDP.PCAP.KD,Data are for areas under the effective control...,NaN
458,SDN,SP.POP.GROW,Data sources : United Nations World Population...,NaN
231,HND,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
481,SOM,NY.GDP.MKTP.PP.KD,Estimates are based on regression.,NaN
496,SRB,SP.POP.TOTL.FE.ZS,Excluding Kosovo.,NaN
199,GHA,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
354,MKD,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
100,CMR,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
542,TLS,NY.GDP.MKTP.PP.KD,Estimates are based on regression.,NaN


<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CountryCode  613 non-null    str    
 1   SeriesCode   613 non-null    str    
 2   DESCRIPTION  613 non-null    str    
 3   Unnamed: 3   0 non-null      float64
dtypes: float64(1), str(3)
memory usage: 19.3 KB


None

In [ ]:
df_footnote.info()

<class 'pandas.DataFrame'>
RangeIndex: 643638 entries, 0 to 643637
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CountryCode  643638 non-null  str    
 1   SeriesCode   643638 non-null  str    
 2   Year         643638 non-null  str    
 3   DESCRIPTION  643638 non-null  str    
 4   Unnamed: 4   0 non-null       float64
dtypes: float64(1), str(4)
memory usage: 24.6 MB


In [ ]:
df_series_code = df_footnote[df_footnote["SeriesCode"].isin(["SP.POP.GROW" , "NOED"])] #attainment
df_series_code

,CountryCode,SeriesCode,Year,DESCRIPTION,Unnamed: 4


In [ ]:
country = df_country_series[df_country_series["CountryCode"] == "DEU"]
country

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
132,DEU,SP.POP.GROW,Data sources : Eurostat,NaN
133,DEU,SP.POP.TOTL,Data sources : Eurostat,NaN


In [ ]:
df_series_code = df_series[df_series["Series Code"].isin(["SP.POP.GROW" , "SP.POP.TOTL"])] 
df_series_code

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
2589,SP.POP.GROW,Health: Population: Dynamics,Population growth (annual %),Annual population growth rate. Population is b...,Annual population growth rate for year t is th...,NaN,Annual,NaN,NaN,Weighted average,...,NaN,NaN,Derived from total population. Population sour...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2590,SP.POP.TOTL,Health: Population: Structure,"Population, total",NaN,Total population is based on the de facto defi...,NaN,Annual,NaN,NaN,Sum,...,NaN,Relevance to gender indicator: disaggregating ...,(1) United Nations Population Division. World ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
display(df_country_series["CountryCode"].value_counts(), sort=True)
# CHI und CHN könnte gleich sein
display(df_country_series["SeriesCode"].value_counts(), sort=True) # 


display(df_series["Series Code"].value_counts(), sort=True)


CountryCode
GEO    18
MDA    18
CYP    12
MAR    12
MUS    12
       ..
WSM     2
YEM     2
ZAF     2
ZMB     2
ZWE     2
Name: count, Length: 211, dtype: int64

SeriesCode
SP.POP.TOTL          211
SP.POP.GROW          211
NY.GDP.PCAP.PP.CD     19
NY.GDP.PCAP.PP.KD     19
NY.GNP.PCAP.PP.CD     19
NY.GDP.MKTP.PP.KD     14
NY.GNP.MKTP.PP.CD     14
NY.GDP.MKTP.PP.CD     14
SP.POP.1564.TO.ZS     13
SP.POP.TOTL.MA.ZS     13
SP.POP.TOTL.FE.ZS     13
SP.POP.0014.TO.ZS     13
NY.GNP.PCAP.CD         6
NY.GDP.PCAP.CD         5
NY.GDP.PCAP.KD         5
SP.POP.1564.MA.IN      4
SP.POP.0014.TO         4
SP.POP.1564.TO         4
SP.POP.1564.FE.IN      4
SP.POP.0014.MA.IN      4
SP.POP.0014.FE.IN      4
Name: count, dtype: int64

Series Code
BAR.NOED.1519.FE.ZS            1
BAR.NOED.1519.ZS               1
BAR.NOED.15UP.FE.ZS            1
BAR.NOED.15UP.ZS               1
BAR.NOED.2024.FE.ZS            1
                              ..
UIS.XUNIT.USCONST.3.FSGOV      1
UIS.XUNIT.USCONST.4.FSGOV      1
UIS.XUNIT.USCONST.56.FSGOV     1
XGDP.23.FSGOV.FDINSTADM.FFD    1
XGDP.56.FSGOV.FDINSTADM.FFD    1
Name: count, Length: 3665, dtype: int64

In [ ]:
# CHI wird fuer Chile stehen und CHN fuer China. CN exestiert nicht.

country_chi = df_country_series[df_country_series["CountryCode"] == "CHI"]
display(country_chi)

country_chn = df_country_series[df_country_series["CountryCode"] == "CHL"]
display(country_chn)

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
84,CHI,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
85,CHI,SP.POP.0014.TO.ZS,"Refers to Guernsey, and Jersey.",NaN
86,CHI,SP.POP.1564.TO.ZS,"Refers to Guernsey, and Jersey.",NaN
87,CHI,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
88,CHI,SP.POP.TOTL.FE.ZS,"Refers to Guernsey, and Jersey.",NaN
89,CHI,SP.POP.TOTL.MA.ZS,"Refers to Guernsey, and Jersey.",NaN


,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
90,CHL,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
91,CHL,SP.POP.TOTL,Data sources : United Nations World Population...,NaN


### 1. EDA


In [9]:
df_combined = df_combined.copy()
df_combined

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
0,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Secondary,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.F,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Secondary,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.GPI,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Secondary,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.M,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Secondary,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arab World,ARB,"Adjusted net enrolment rate, primary, both sex...",SE.PRM.TENR,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Primary,...,82.685509,83.280342,84.011871,84.195961,85.211998,85.24514,86.101669,85.51194,85.320152,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886925,Zimbabwe,ZWE,"Youth illiterate population, 15-24 years, male...",UIS.LP.AG15T24.M,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,NaN,...,NaN,NaN,NaN,NaN,NaN,166679.00000,NaN,NaN,199464.000000,NaN
886926,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, b...",SE.ADT.1524.LT.ZS,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Literacy,...,NaN,NaN,NaN,NaN,NaN,90.93070,NaN,NaN,90.428120,NaN
886927,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, f...",SE.ADT.1524.LT.FE.ZS,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Literacy,...,NaN,NaN,NaN,NaN,NaN,92.12456,NaN,NaN,93.188350,NaN
886928,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, g...",SE.ADT.1524.LT.FM.ZS,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Literacy,...,NaN,NaN,NaN,NaN,NaN,1.02828,NaN,NaN,1.063890,NaN


In [11]:
# Chosen Indicatos:
# SE.XPD.TOTL.GD.ZS -> Government expenditure on education as % of GDP (%)
# SE.XPD.TOTL.GB.ZS -> Expenditure on education as % of total government expenditure (%)

# NY.GDP.MKTP.PP.KD -> GDP, PPP (constant 2011 international $)
# NY.GDP.PCAP.PP.KD -> GDP per capita, PPP (constant 2011 international $)

# UIS.SAP.1.G1.M -> Population of the official entrance age to primary education, male (number)
# UIS.SAP.1.G1.F -> Population of the official entrance age to primary education, female (number)

# UIS.GER.123.F -> Gross enrolment ratio, primary and secondary, females (%)
# UIS.GER.123.M -> Gross enrolment ratio, primary and secondary, males (%)

# UIS.NER.1.GPI -> Net enrolment rate, primary, gender parity index (GPI)
# UIS.NER.23.GPI -> Net enrolment rate, secondary, gender parity index (GPI)

# SE.PRM.CMPT.FE.ZS -> Primary completion rate, female (%)
# SE.PRM.CMPT.MA.ZS -> Primary completion rate, male (%)

selected_codes = [
    "SE.XPD.TOTL.GD.ZS",
    "SE.XPD.TOTL.GB.ZS",
    "NY.GDP.MKTP.PP.KD",
    "NY.GDP.PCAP.PP.KD",
    "UIS.SAP.1.G1.M",
    "UIS.SAP.1.G1.F",
    "UIS.GER.123.F",
    "UIS.GER.123.M",
    "UIS.NER.1.GPI",
    "UIS.NER.23.GPI",
    "SE.PRM.CMPT.FE.ZS",
    "SE.PRM.CMPT.MA.ZS"]

In [12]:
df_indicator_data = df_combined.loc[(df_combined["Indicator Code"].isin(selected_codes))]
df_indicator_data

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
1224,Arab World,ARB,Expenditure on education as % of total governm...,SE.XPD.TOTL.GB.ZS,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Expenditures,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1246,Arab World,ARB,"GDP per capita, PPP (constant 2011 internation...",NY.GDP.PCAP.PP.KD,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Economic Policy & Debt: Purchasing power parity,...,1.343403e+04,1.386042e+04,1.437783e+04,1.435481e+04,1.475905e+04,1.482591e+04,1.534280e+04,1.545497e+04,NaN,NaN
1248,Arab World,ARB,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Economic Policy & Debt: Purchasing power parity,...,4.260279e+12,4.500388e+12,4.779549e+12,4.882776e+12,5.132269e+12,5.265230e+12,5.561251e+12,5.714653e+12,NaN,NaN
1260,Arab World,ARB,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Expenditures,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1324,Arab World,ARB,"Gross enrolment ratio, primary and secondary, ...",UIS.GER.123.F,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Secondary,...,7.694061e+01,7.665801e+01,7.747315e+01,7.841426e+01,7.945959e+01,8.052322e+01,8.216338e+01,8.091618e+01,81.176811,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884921,Zimbabwe,ZWE,"Net enrolment rate, secondary, gender parity i...",UIS.NER.23.GPI,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Secondary,...,NaN,NaN,NaN,NaN,NaN,NaN,1.002280e+00,1.011080e+00,NaN,NaN
885601,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.F,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Population,...,1.784010e+05,1.803770e+05,1.832270e+05,1.868440e+05,1.910540e+05,1.957320e+05,2.015990e+05,2.082110e+05,215118.000000,221749.0
885602,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.M,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Population,...,1.793050e+05,1.813000e+05,1.841870e+05,1.878030e+05,1.919210e+05,1.965500e+05,2.024060e+05,2.090290e+05,215964.000000,222623.0
885801,Zimbabwe,ZWE,"Primary completion rate, female (%)",SE.PRM.CMPT.FE.ZS,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Primary,...,NaN,NaN,NaN,NaN,NaN,NaN,9.035918e+01,9.093144e+01,NaN,NaN


In [ ]:
display(df_indicator_data["Indicator Code"].value_counts())
display(df_indicator_data["Indicator Code"].unique())

Indicator Code
SE.XPD.TOTL.GB.ZS    242
NY.GDP.PCAP.PP.KD    242
NY.GDP.MKTP.PP.KD    242
SE.XPD.TOTL.GD.ZS    242
UIS.GER.123.F        242
UIS.GER.123.M        242
UIS.NER.1.GPI        242
UIS.NER.23.GPI       242
UIS.SAP.1.G1.F       242
UIS.SAP.1.G1.M       242
SE.PRM.CMPT.FE.ZS    242
SE.PRM.CMPT.MA.ZS    242
Name: count, dtype: int64

<StringArray>
['SE.XPD.TOTL.GB.ZS', 'NY.GDP.PCAP.PP.KD', 'NY.GDP.MKTP.PP.KD',
 'SE.XPD.TOTL.GD.ZS',     'UIS.GER.123.F',     'UIS.GER.123.M',
     'UIS.NER.1.GPI',    'UIS.NER.23.GPI',    'UIS.SAP.1.G1.F',
    'UIS.SAP.1.G1.M', 'SE.PRM.CMPT.FE.ZS', 'SE.PRM.CMPT.MA.ZS']
Length: 12, dtype: str

In [ ]:
df_indicator_data.isnull().sum()

Country Name             0
Country Code             0
Indicator Name           0
Indicator Code           0
Currency Unit          324
Special Notes         1164
Region                 336
Income Group           336
System of trade        504
Topic                    0
Short definition      2662
Long definition          0
Periodicity           2420
Other notes           2662
Aggregation method    2420
Source                   0
1999                   997
2000                   982
2001                   986
2002                   948
2003                   996
2004                   940
2005                   942
2006                   968
2007                   935
2008                   903
2009                   899
2010                   912
2011                   902
2012                   939
2013                   973
2014                  1126
2015                  1784
dtype: int64

In [ ]:
df_indicator_data["Country Name"].value_counts() # 12 * 242 -> 12 Indicator for every Country/Continent/Special-Grouping

Country Name
Arab World                                     12
East Asia & Pacific                            12
East Asia & Pacific (excluding high income)    12
Euro area                                      12
Europe & Central Asia                          12
                                               ..
Virgin Islands (U.S.)                          12
West Bank and Gaza                             12
Yemen, Rep.                                    12
Zambia                                         12
Zimbabwe                                       12
Name: count, Length: 242, dtype: int64

In [13]:
year_columns = [str(year) for year in DATE_RANGE_INDEX]
year_columns

['1999',
 '2000',
 '2001',
 '2002',
 '2003',
 '2004',
 '2005',
 '2006',
 '2007',
 '2008',
 '2009',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015']

In [14]:

df_num = df_indicator_data.select_dtypes(include="number")

df_indicator_data = df_indicator_data.dropna(
    axis="index",
    thresh=len(year_columns)*0.2,
    subset=df_num.columns
)# Empty year columns with 20% emptry rows, will be deleted

In [ ]:
df_indicator_data.info()

<class 'pandas.DataFrame'>
Index: 2299 entries, 1246 to 885803
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country Name        2299 non-null   str    
 1   Country Code        2299 non-null   str    
 2   Indicator Name      2299 non-null   str    
 3   Indicator Code      2299 non-null   str    
 4   Currency Unit       2044 non-null   str    
 5   Special Notes       1474 non-null   str    
 6   Region              2044 non-null   str    
 7   Income Group        2044 non-null   str    
 8   System of trade     1996 non-null   str    
 9   Topic               2299 non-null   str    
 10  Short definition    161 non-null    str    
 11  Long definition     2299 non-null   str    
 12  Periodicity         430 non-null    str    
 13  Other notes         161 non-null    str    
 14  Aggregation method  430 non-null    str    
 15  Source              2299 non-null   str    
 16  1999             

In [15]:
#df = df_indicator_data["Indicator Code"].str.contains("NY.GDP.MKTP.PP.KD")
year_columns

#df  = df_indicator_data.loc[(df_indicator_data["Indicator Code"] == "NY.GDP.MKTP.PP.KD") & (df_indicator_data["Country Code"] == "DEU")]
df = df_indicator_data.loc[(df_indicator_data["Indicator Code"] == "NY.GDP.MKTP.PP.KD")]
df_indicator_data["Median_GDP_PPP"] = df[year_columns].median(axis=1)

df_indicator_data

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2007,2008,2009,2010,2011,2012,2013,2014,2015,Median_GDP_PPP
1246,Arab World,ARB,"GDP per capita, PPP (constant 2011 internation...",NY.GDP.PCAP.PP.KD,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Economic Policy & Debt: Purchasing power parity,...,1.386042e+04,1.437783e+04,1.435481e+04,1.475905e+04,1.482591e+04,1.534280e+04,1.545497e+04,NaN,NaN,NaN
1248,Arab World,ARB,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Economic Policy & Debt: Purchasing power parity,...,4.500388e+12,4.779549e+12,4.882776e+12,5.132269e+12,5.265230e+12,5.561251e+12,5.714653e+12,NaN,NaN,4.260279e+12
1324,Arab World,ARB,"Gross enrolment ratio, primary and secondary, ...",UIS.GER.123.F,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Secondary,...,7.665801e+01,7.747315e+01,7.841426e+01,7.945959e+01,8.052322e+01,8.216338e+01,8.091618e+01,81.176811,NaN,NaN
1326,Arab World,ARB,"Gross enrolment ratio, primary and secondary, ...",UIS.GER.123.M,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Secondary,...,8.330459e+01,8.397731e+01,8.492591e+01,8.589872e+01,8.714454e+01,8.831270e+01,8.690956e+01,87.240860,NaN,NaN
1652,Arab World,ARB,"Net enrolment rate, primary, gender parity ind...",UIS.NER.1.GPI,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Primary,...,9.391200e-01,9.425900e-01,9.409800e-01,9.443100e-01,9.445400e-01,9.606300e-01,9.620100e-01,0.964480,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884921,Zimbabwe,ZWE,"Net enrolment rate, secondary, gender parity i...",UIS.NER.23.GPI,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Secondary,...,NaN,NaN,NaN,NaN,NaN,1.002280e+00,1.011080e+00,NaN,NaN,NaN
885601,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.F,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Population,...,1.803770e+05,1.832270e+05,1.868440e+05,1.910540e+05,1.957320e+05,2.015990e+05,2.082110e+05,215118.000000,221749.0,NaN
885602,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.M,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Population,...,1.813000e+05,1.841870e+05,1.878030e+05,1.919210e+05,1.965500e+05,2.024060e+05,2.090290e+05,215964.000000,222623.0,NaN
885801,Zimbabwe,ZWE,"Primary completion rate, female (%)",SE.PRM.CMPT.FE.ZS,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Primary,...,NaN,NaN,NaN,NaN,NaN,9.035918e+01,9.093144e+01,NaN,NaN,NaN


In [16]:
# Only countries seem to be interesting enough to keep. No use in comparing continents or groupings of countries.
continents_and_groupings = [
    "World",
    "High income",
    "OECD members",
    "Middle income",
    "Upper middle income",
    "Europe & Central Asia",
    "East Asia & Pacific",
    "European Union",
    "North America",
    "East Asia & Pacific (excluding high income)",
    "Euro area",
    "Lower middle income",
    "Latin America & Caribbean",
    "Latin America & Caribbean (excluding high income)",
    "Europe & Central Asia (excluding high income)",
    "Middle East & North Africa",
    "South Asia",
    "Arab World",
    "Middle East & North Africa (excluding high income)",
    "Sub-Saharan Africa",
    "Sub-Saharan Africa (excluding high income)",
    "Least developed countries: UN classification",
    "Heavily indebted poor countries (HIPC)",
    "Low income",
    "South Africa",
    "Central African Republic",
    "Low & middle income"]

df_countries_data = df_indicator_data.loc[(~df_indicator_data["Country Name"].isin(continents_and_groupings))]
df_countries_data

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2007,2008,2009,2010,2011,2012,2013,2014,2015,Median_GDP_PPP
92849,Afghanistan,AFG,Expenditure on education as % of total governm...,SE.XPD.TOTL.GB.ZS,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,General trade system,Expenditures,...,NaN,NaN,NaN,1.706756e+01,1.604843e+01,1.035680e+01,1.410280e+01,1.446593e+01,1.250900e+01,NaN
92871,Afghanistan,AFG,"GDP per capita, PPP (constant 2011 internation...",NY.GDP.PCAP.PP.KD,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,General trade system,Economic Policy & Debt: Purchasing power parity,...,1.284775e+03,1.298143e+03,1.531174e+03,1.614255e+03,1.660740e+03,1.839274e+03,1.814156e+03,1.780382e+03,1.747978e+03,NaN
92873,Afghanistan,AFG,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,General trade system,Economic Policy & Debt: Purchasing power parity,...,3.419659e+10,3.543156e+10,4.287950e+10,4.649566e+10,4.933825e+10,5.646010e+10,5.756623e+10,5.832180e+10,5.897066e+10,3.915553e+10
92885,Afghanistan,AFG,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,General trade system,Expenditures,...,NaN,NaN,NaN,3.461490e+00,3.439140e+00,2.527540e+00,3.481400e+00,3.780280e+00,3.317540e+00,NaN
92949,Afghanistan,AFG,"Gross enrolment ratio, primary and secondary, ...",UIS.GER.123.F,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,General trade system,Secondary,...,5.211268e+01,5.732527e+01,5.910382e+01,6.358303e+01,6.498535e+01,6.856315e+01,6.795231e+01,6.799893e+01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884921,Zimbabwe,ZWE,"Net enrolment rate, secondary, gender parity i...",UIS.NER.23.GPI,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Secondary,...,NaN,NaN,NaN,NaN,NaN,1.002280e+00,1.011080e+00,NaN,NaN,NaN
885601,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.F,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Population,...,1.803770e+05,1.832270e+05,1.868440e+05,1.910540e+05,1.957320e+05,2.015990e+05,2.082110e+05,2.151180e+05,2.217490e+05,NaN
885602,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.M,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Population,...,1.813000e+05,1.841870e+05,1.878030e+05,1.919210e+05,1.965500e+05,2.024060e+05,2.090290e+05,2.159640e+05,2.226230e+05,NaN
885801,Zimbabwe,ZWE,"Primary completion rate, female (%)",SE.PRM.CMPT.FE.ZS,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Primary,...,NaN,NaN,NaN,NaN,NaN,9.035918e+01,9.093144e+01,NaN,NaN,NaN


In [49]:
ger_gdp = df_countries_data.loc[(df_countries_data["Country Name"] == "Germany") & (df_countries_data["Median_GDP_PPP"].notna()),
"Median_GDP_PPP"].iloc[0]
ger_gdp

np.float64(3224475116048.83)

In [50]:
(ger_gdp / 10**10) * MIN_GDP_RATIO_TO_GERMANY

np.float64(32.2447511604883)

In [45]:
666432522053.348 / 10**10

66.6432522053348

In [42]:
666432522053.348 - ger_gdp* 0.2

np.float64(21537498843.58203)

In [ ]:
666432522053.348 * 0.2

133286504410.66962

In [54]:
df_ger_gdp = df_countries_data.iloc[(df_countries_data["Median_GDP_PPP"] >= ger_gdp * MIN_GDP_RATIO_TO_GERMANY)]
df_ger_gdp["Country Name"]


100203                 Algeria
118528               Argentina
129523               Australia
133188                 Austria
158843                 Belgium
188163                  Brazil
221148                  Canada
243138                   China
246803                Colombia
305443        Egypt, Arab Rep.
338428                  France
356753                 Germany
415393                   India
419058               Indonesia
422723      Iran, Islamic Rep.
426388                    Iraq
441048                   Italy
448378                   Japan
470368             Korea, Rep.
532673                Malaysia
558328                  Mexico
602308             Netherlands
620633                 Nigeria
635293                Pakistan
657283             Philippines
660948                  Poland
675608                 Romania
679273      Russian Federation
697598            Saudi Arabia
745243                   Spain
778228                  Sweden
781893             Switzerland
796553  

In [68]:
df_gdp = df_countries_data.loc[(df_countries_data["Country Name"].isin(df_ger_gdp["Country Name"]))]
df_gdp

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2007,2008,2009,2010,2011,2012,2013,2014,2015,Median_GDP_PPP
100201,Algeria,DZA,"GDP per capita, PPP (constant 2011 internation...",NY.GDP.PCAP.PP.KD,Algerian dinar,NaN,Middle East & North Africa,Upper middle income,Special trade system,Economic Policy & Debt: Purchasing power parity,...,1.257065e+04,1.266040e+04,1.264754e+04,1.287060e+04,1.299035e+04,1.316196e+04,1.325362e+04,1.348334e+04,1.372472e+04,NaN
100203,Algeria,DZA,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Algerian dinar,NaN,Middle East & North Africa,Upper middle income,Special trade system,Economic Policy & Debt: Purchasing power parity,...,4.311744e+11,4.413507e+11,4.485546e+11,4.648558e+11,4.782988e+11,4.944402e+11,5.081246e+11,5.273780e+11,5.472257e+11,4.311744e+11
100279,Algeria,DZA,"Gross enrolment ratio, primary and secondary, ...",UIS.GER.123.F,Algerian dinar,NaN,Middle East & North Africa,Upper middle income,Special trade system,Secondary,...,8.654514e+01,8.939616e+01,9.904778e+01,1.035571e+02,1.058897e+02,NaN,NaN,NaN,NaN,NaN
100281,Algeria,DZA,"Gross enrolment ratio, primary and secondary, ...",UIS.GER.123.M,Algerian dinar,NaN,Middle East & North Africa,Upper middle income,Special trade system,Secondary,...,8.651773e+01,9.078054e+01,1.013917e+02,1.045367e+02,1.068557e+02,NaN,NaN,NaN,NaN,NaN
100607,Algeria,DZA,"Net enrolment rate, primary, gender parity ind...",UIS.NER.1.GPI,Algerian dinar,NaN,Middle East & North Africa,Upper middle income,Special trade system,Primary,...,9.790900e-01,9.807500e-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866200,Vietnam,VNM,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,Vietnamese dong,Based on data from the Vietnam Statistics Offi...,East Asia & Pacific,Lower middle income,General trade system,Expenditures,...,NaN,4.874810e+00,4.814240e+00,5.132060e+00,4.813510e+00,5.525490e+00,5.658730e+00,NaN,NaN,NaN
867276,Vietnam,VNM,Population of the official entrance age to pri...,UIS.SAP.1.G1.F,Vietnamese dong,Based on data from the Vietnam Statistics Offi...,East Asia & Pacific,Lower middle income,General trade system,Population,...,6.333750e+05,6.264830e+05,6.303410e+05,6.397550e+05,6.466720e+05,6.545310e+05,6.652020e+05,6.774380e+05,6.901350e+05,NaN
867277,Vietnam,VNM,Population of the official entrance age to pri...,UIS.SAP.1.G1.M,Vietnamese dong,Based on data from the Vietnam Statistics Offi...,East Asia & Pacific,Lower middle income,General trade system,Population,...,6.697260e+05,6.632610e+05,6.681270e+05,6.792000e+05,6.885780e+05,7.003830e+05,7.170030e+05,7.361930e+05,7.558920e+05,NaN
867476,Vietnam,VNM,"Primary completion rate, female (%)",SE.PRM.CMPT.FE.ZS,Vietnamese dong,Based on data from the Vietnam Statistics Offi...,East Asia & Pacific,Lower middle income,General trade system,Primary,...,NaN,NaN,NaN,NaN,NaN,NaN,9.927908e+01,1.082595e+02,1.039037e+02,NaN


In [69]:
df_gdp["Country Name"].value_counts()

Country Name
Argentina               12
Belgium                 12
Colombia                12
Indonesia               12
Iran, Islamic Rep.      12
Italy                   12
Japan                   12
Mexico                  12
Netherlands             12
Pakistan                12
Philippines             12
Poland                  12
Spain                   12
Sweden                  12
Switzerland             12
Thailand                12
Turkey                  12
Ukraine                 12
Egypt, Arab Rep.        11
India                   11
Korea, Rep.             11
Romania                 11
Saudi Arabia            11
Australia               10
Austria                 10
Brazil                  10
France                  10
Germany                 10
Iraq                    10
United Kingdom          10
United States           10
Venezuela, RB           10
Algeria                  9
Nigeria                  9
Russian Federation       9
Canada                   8
China          

In [70]:
df_gdp.info()

<class 'pandas.DataFrame'>
Index: 425 entries, 100201 to 867478
Data columns (total 34 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country Name        425 non-null    str    
 1   Country Code        425 non-null    str    
 2   Indicator Name      425 non-null    str    
 3   Indicator Code      425 non-null    str    
 4   Currency Unit       425 non-null    str    
 5   Special Notes       294 non-null    str    
 6   Region              425 non-null    str    
 7   Income Group        425 non-null    str    
 8   System of trade     415 non-null    str    
 9   Topic               425 non-null    str    
 10  Short definition    34 non-null     str    
 11  Long definition     425 non-null    str    
 12  Periodicity         80 non-null     str    
 13  Other notes         34 non-null     str    
 14  Aggregation method  80 non-null     str    
 15  Source              425 non-null    str    
 16  1999            

In [72]:
df_gdp.sample(10)

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2007,2008,2009,2010,2011,2012,2013,2014,2015,Median_GDP_PPP
844198,United Kingdom,GBR,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Pound sterling,NaN,Europe & Central Asia,High income: OECD,General trade system,Economic Policy & Debt: Purchasing power parity,...,2.353817e+12,2.342694e+12,2.244588e+12,2.282623e+12,2.315781e+12,2.350086e+12,2.398319e+12,2.471569e+12,2.529551e+12,2.282623e+12
746533,Spain,ESP,"Primary completion rate, male (%)",SE.PRM.CMPT.MA.ZS,Euro,A simple multiplier is used to convert the nat...,Europe & Central Asia,High income: OECD,Special trade system,Primary,...,9.844452e+01,9.983423e+01,1.022561e+02,1.039538e+02,1.021105e+02,1.025546e+02,9.922102e+01,9.770460e+01,9.886059e+01,NaN
840937,United Arab Emirates,ARE,"Net enrolment rate, primary, gender parity ind...",UIS.NER.1.GPI,U.A.E. dirham,April 2013 database update: Based on data from...,Middle East & North Africa,High income: nonOECD,General trade system,Primary,...,1.029090e+00,NaN,1.027000e+00,1.034110e+00,1.005250e+00,1.020340e+00,1.018900e+00,9.989800e-01,NaN,NaN
338404,France,FRA,Expenditure on education as % of total governm...,SE.XPD.TOTL.GB.ZS,Euro,A simple multiplier is used to convert the nat...,Europe & Central Asia,High income: OECD,Special trade system,Expenditures,...,1.042599e+01,1.027361e+01,1.010988e+01,1.006913e+01,9.863940e+00,9.736760e+00,9.661880e+00,NaN,NaN,NaN
867277,Vietnam,VNM,Population of the official entrance age to pri...,UIS.SAP.1.G1.M,Vietnamese dong,Based on data from the Vietnam Statistics Offi...,East Asia & Pacific,Lower middle income,General trade system,Population,...,6.697260e+05,6.632610e+05,6.681270e+05,6.792000e+05,6.885780e+05,7.003830e+05,7.170030e+05,7.361930e+05,7.558920e+05,NaN
778636,Sweden,SWE,"Net enrolment rate, secondary, gender parity i...",UIS.NER.23.GPI,Swedish krona,Fiscal year end: June 30; reporting period for...,Europe & Central Asia,High income: OECD,General trade system,Secondary,...,9.997100e-01,1.001230e+00,1.000340e+00,1.000330e+00,1.001240e+00,9.995200e-01,9.960600e-01,9.966500e-01,NaN,NaN
247892,Colombia,COL,Population of the official entrance age to pri...,UIS.SAP.1.G1.M,Colombian peso,NaN,Latin America & Caribbean,Upper middle income,General trade system,Population,...,4.152690e+05,4.167130e+05,4.178340e+05,4.168830e+05,4.145030e+05,4.103790e+05,4.051620e+05,3.995500e+05,3.945330e+05,NaN
419070,Indonesia,IDN,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,Indonesian rupiah,Fiscal year end: March 31; reporting period fo...,East Asia & Pacific,Lower middle income,Special trade system,Expenditures,...,3.044940e+00,2.901130e+00,3.522740e+00,2.813970e+00,3.190150e+00,3.405780e+00,3.357680e+00,3.278610e+00,3.591810e+00,NaN
100203,Algeria,DZA,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Algerian dinar,NaN,Middle East & North Africa,Upper middle income,Special trade system,Economic Policy & Debt: Purchasing power parity,...,4.311744e+11,4.413507e+11,4.485546e+11,4.648558e+11,4.782988e+11,4.944402e+11,5.081246e+11,5.273780e+11,5.472257e+11,4.311744e+11
680361,Russian Federation,RUS,Population of the official entrance age to pri...,UIS.SAP.1.G1.F,Russian ruble,NaN,Europe & Central Asia,High income: nonOECD,General trade system,Population,...,6.136560e+05,6.158820e+05,6.288680e+05,6.488780e+05,6.730600e+05,6.991450e+05,7.200360e+05,7.381420e+05,7.557130e+05,NaN


### 1.1 Plots


In [ ]:
year_columns = [str(year) for year in DATE_RANGE_INDEX]
year_columns

['1999',
 '2000',
 '2001',
 '2002',
 '2003',
 '2004',
 '2005',
 '2006',
 '2007',
 '2008',
 '2009',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015']

In [ ]:
#df_data[df_data["Country Code"] == "USA"]
df_filled_data = df_data.loc[(df_data[year_columns].notna().all(axis=1)) & (df_data["Country Code"] == "USA") & (df_data["Indicator Name"].str.contains("population", case=False))]
df_filled_data

,Country Name,Country Code,Indicator Name,Indicator Code,1999,2000,2001,2002,2003,2004,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
848922,United States,USA,Population growth (annual %),SP.POP.GROW,1.148340e+00,1.112769e+00,9.897414e-01,9.277975e-01,8.594817e-01,9.254840e-01,...,9.642539e-01,9.510552e-01,9.458653e-01,8.766513e-01,8.364215e-01,7.456144e-01,7.464199e-01,7.002623e-01,7.431243e-01,7.297320e-01
848926,United States,USA,Population of the official age for lower secon...,SP.SEC.LTOT.IN,1.201333e+07,1.216902e+07,1.231032e+07,1.245034e+07,1.257160e+07,1.267502e+07,...,1.285246e+07,1.282366e+07,1.274381e+07,1.265000e+07,1.257070e+07,1.252428e+07,1.237527e+07,1.227283e+07,1.225365e+07,1.229571e+07
848927,United States,USA,Population of the official age for lower secon...,SP.SEC.LTOT.FE.IN,5.848454e+06,5.920295e+06,5.988669e+06,6.050160e+06,6.102527e+06,6.149704e+06,...,6.243729e+06,6.235018e+06,6.196619e+06,6.150355e+06,6.112201e+06,6.091334e+06,6.025577e+06,5.981936e+06,5.978886e+06,6.006711e+06
848928,United States,USA,Population of the official age for lower secon...,SP.SEC.LTOT.MA.IN,6.164874e+06,6.248725e+06,6.321647e+06,6.400175e+06,6.469078e+06,6.525313e+06,...,6.608730e+06,6.588640e+06,6.547190e+06,6.499641e+06,6.458498e+06,6.432949e+06,6.349689e+06,6.290897e+06,6.274767e+06,6.288997e+06
848929,United States,USA,Population of the official age for post-second...,UIS.SAP.4,3.871212e+06,3.950141e+06,4.035026e+06,4.111480e+06,4.164915e+06,4.202868e+06,...,4.286293e+06,4.294387e+06,4.312086e+06,4.337900e+06,4.367156e+06,4.403201e+06,4.473879e+06,4.487510e+06,4.424384e+06,4.327800e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849144,United States,USA,"Population, female",SP.POP.TOTL.FE.IN,1.416695e+08,1.431905e+08,1.445523e+08,1.458402e+08,1.470439e+08,1.483620e+08,...,1.511100e+08,1.525269e+08,1.539522e+08,1.552808e+08,1.565514e+08,1.576812e+08,1.588139e+08,1.598773e+08,1.610179e+08,1.621492e+08
849145,United States,USA,"Population, female (% of total)",SP.POP.TOTL.FE.ZS,5.077031e+01,5.074753e+01,5.072563e+01,5.070496e+01,5.068594e+01,5.066915e+01,...,5.064349e+01,5.063450e+01,5.062653e+01,5.061774e+01,5.060687e+01,5.059343e+01,5.057792e+01,5.056131e+01,5.054500e+01,5.053003e+01
849146,United States,USA,"Population, male",SP.POP.TOTL.MA.IN,1.373705e+08,1.389719e+08,1.404166e+08,1.417850e+08,1.430640e+08,1.444433e+08,...,1.472699e+08,1.487043e+08,1.501417e+08,1.514907e+08,1.527968e+08,1.539822e+08,1.551845e+08,1.563276e+08,1.575455e+08,1.587475e+08
849147,United States,USA,"Population, male (% of total)",SP.POP.TOTL.MA.ZS,4.922969e+01,4.925247e+01,4.927437e+01,4.929504e+01,4.931406e+01,4.933085e+01,...,4.935651e+01,4.936550e+01,4.937347e+01,4.938226e+01,4.939313e+01,4.940657e+01,4.942208e+01,4.943869e+01,4.945500e+01,4.946997e+01


In [ ]:
# SP.POP.1018.FE.UN = Population, ages 10-18, female
# SP.POP.1018.MA.UN = Population, ages 10-18, male
df_data[df_data["Country Code"] == "USA"]
df_filled_data = df_data.loc[(df_data["1999"] >= 1 ) & (df_data["Country Code"] == "USA") & (df_data["Indicator Name"].str.contains("male", case=False))]
df_filled_data

,Country Name,Country Code,Indicator Name,Indicator Code,1999,2000,2001,2002,2003,2004,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
846620,United States,USA,"Adjusted net enrolment rate, primary, female (%)",SE.PRM.TENR.FE,9.779105e+01,9.773678e+01,98.037827,9.679218e+01,96.170967,9.514385e+01,...,9.725755e+01,9.785545e+01,9.794343e+01,9.709736e+01,95.138184,9.467457e+01,9.410391e+01,9.381774e+01,9.443206e+01,94.765541
846622,United States,USA,"Adjusted net enrolment rate, primary, male (%)",SE.PRM.TENR.MA,9.725062e+01,9.750291e+01,96.620857,9.525357e+01,94.931084,9.376343e+01,...,9.529584e+01,9.652141e+01,9.704943e+01,9.501399e+01,94.468750,9.324649e+01,9.411461e+01,9.383361e+01,9.365279e+01,94.158813
846628,United States,USA,Adjusted net intake rate to Grade 1 of primary...,UIS.NIRA.1.F,8.803209e+01,8.423761e+01,85.486191,8.337826e+01,84.070671,7.939470e+01,...,8.290853e+01,8.766076e+01,9.020750e+01,8.369993e+01,82.832977,8.687901e+01,7.739850e+01,7.790285e+01,7.978172e+01,NaN
846630,United States,USA,Adjusted net intake rate to Grade 1 of primary...,UIS.NIRA.1.M,8.514682e+01,8.676923e+01,78.982811,7.753821e+01,81.926620,7.704565e+01,...,7.985333e+01,8.265864e+01,8.379000e+01,7.682121e+01,82.588860,8.253268e+01,7.760338e+01,7.564341e+01,7.733834e+01,NaN
847737,United States,USA,"Enrolment in early childhood education, female...",UIS.E.0.F,3.446666e+06,3.494559e+06,NaN,3.797312e+06,NaN,3.544902e+06,...,3.587208e+06,3.625338e+06,3.494297e+06,3.490333e+06,NaN,4.346524e+06,4.416691e+06,4.263450e+06,4.300430e+06,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849704,United States,USA,"Total inbound internationally mobile students,...",UIS.MS.56.F,1.893600e+05,1.990960e+05,199096.000000,2.555930e+05,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.593570e+05,3.815390e+05,NaN
849710,United States,USA,"Total net enrolment rate, primary, female (%)",UIS.NERT.1.F,9.969056e+01,9.943364e+01,100.000000,9.821479e+01,97.547127,9.730418e+01,...,9.947161e+01,1.000000e+02,9.982290e+01,9.921085e+01,97.704102,9.707460e+01,9.693871e+01,9.596595e+01,9.802072e+01,NaN
849712,United States,USA,"Total net enrolment rate, primary, male (%)",UIS.NERT.1.M,9.952068e+01,9.946889e+01,98.953949,9.791414e+01,96.911438,9.643413e+01,...,9.800125e+01,9.932148e+01,9.970499e+01,9.774106e+01,97.430099,9.592931e+01,9.764005e+01,9.687037e+01,9.765747e+01,NaN
849961,United States,USA,"Unemployment, female (% of female labor force)",SL.UEM.TOTL.FE.ZS,4.300000e+00,4.100000e+00,4.600000,5.600000e+00,5.700000,5.400000e+00,...,4.600000e+00,4.500000e+00,5.400000e+00,8.100000e+00,8.600000,8.400000e+00,7.900000e+00,7.100000e+00,6.100000e+00,5.200000


In [ ]:
df_data[df_data["Country Code"] == "USA"]
df_filled_data = df_data.loc[(df_data["2000"] >= 0) & (df_data["Country Code"] == "USA") & (df_data["Indicator Code"].str.contains("uis.ger", case=False))]
df_filled_data

,Country Name,Country Code,Indicator Name,Indicator Code,1999,2000,2001,2002,2003,2004,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
847924,United States,USA,"Gross enrolment ratio, lower secondary, gender...",UIS.GER.2.GPI,NaN,0.995930,0.996270,0.981290,1.001160,1.044020,...,0.993210,1.009030,0.998390,1.005550,1.014270,0.995430,1.009000,0.987660,1.004710,NaN
847926,United States,USA,"Gross enrolment ratio, post-secondary non-tert...",UIS.GER.4,42.444180,39.331379,41.450119,11.204090,10.163860,10.207100,...,10.132980,10.399710,9.808620,10.871180,12.344760,12.990500,12.587420,26.171120,25.937309,NaN
847927,United States,USA,"Gross enrolment ratio, post-secondary non-tert...",UIS.GER.4.F,50.338001,47.387451,49.007561,14.084080,13.874940,14.475730,...,14.358350,14.720400,13.723610,15.260180,16.808399,17.650419,17.545521,35.092339,34.553951,NaN
847928,United States,USA,"Gross enrolment ratio, post-secondary non-tert...",UIS.GER.4.GPI,1.437060,1.492870,1.427670,1.659790,2.082230,2.338870,...,2.320650,2.316070,2.237520,2.264210,2.064790,2.050570,2.213450,1.973230,1.938050,NaN
847929,United States,USA,"Gross enrolment ratio, post-secondary non-tert...",UIS.GER.4.M,35.028419,31.742590,34.326839,8.485440,6.663510,6.189190,...,6.187200,6.355780,6.133400,6.739730,8.140490,8.607570,7.926780,17.784220,17.829241,NaN
847932,United States,USA,"Gross enrolment ratio, pre-primary, gender par...",UIS.GER.0.GPI,0.966720,1.013320,NaN,1.028150,NaN,0.953320,...,0.997150,0.972700,0.984670,0.980560,1.032870,0.969440,0.968170,0.995640,1.004830,NaN
847934,United States,USA,"Gross enrolment ratio, primary and lower secon...",UIS.GER.12,102.970161,102.138702,102.623451,100.746529,101.377220,101.168053,...,101.581306,102.296333,102.507919,101.757637,100.181252,99.608780,99.707680,100.092850,100.337173,NaN
847935,United States,USA,"Gross enrolment ratio, primary and lower secon...",UIS.GER.12.F,NaN,101.599182,102.553932,100.809158,101.572617,101.876488,...,101.784843,102.438293,102.640259,102.082489,100.146378,99.703690,99.326881,99.536377,100.213760,NaN
847936,United States,USA,"Gross enrolment ratio, primary and lower secon...",UIS.GER.12.GPI,NaN,0.989750,0.998680,1.001210,1.003760,1.013730,...,1.003910,1.002710,1.002520,1.006250,0.999320,1.001860,0.992560,0.989180,0.997590,NaN
847937,United States,USA,"Gross enrolment ratio, primary and lower secon...",UIS.GER.12.M,NaN,102.650887,102.689468,100.687119,101.192001,100.496468,...,101.387970,102.161392,102.382057,101.448540,100.214447,99.518311,100.071404,100.625221,100.455368,NaN


In [ ]:
display(df_indicator_data.info())
display(df_indicator_data.sample(10))

<class 'pandas.DataFrame'>
Index: 215 entries, 1248 to 884513
Data columns (total 34 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country Name        215 non-null    str    
 1   Country Code        215 non-null    str    
 2   Indicator Name      215 non-null    str    
 3   Indicator Code      215 non-null    str    
 4   Currency Unit       190 non-null    str    
 5   Special Notes       143 non-null    str    
 6   Region              190 non-null    str    
 7   Income Group        190 non-null    str    
 8   System of trade     184 non-null    str    
 9   Topic               215 non-null    str    
 10  Short definition    0 non-null      str    
 11  Long definition     215 non-null    str    
 12  Periodicity         215 non-null    str    
 13  Other notes         0 non-null      str    
 14  Aggregation method  215 non-null    str    
 15  Source              215 non-null    str    
 16  1999              

None

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2007,2008,2009,2010,2011,2012,2013,2014,2015,Median
210153,Cabo Verde,CPV,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Cabo Verde escudo,Cabo Verde is the new name for the country pre...,Sub-Saharan Africa,Lower middle income,General trade system,Economic Policy & Debt: Purchasing power parity,...,2.740559e+09,2.922821e+09,2.885688e+09,2.928015e+09,3.044225e+09,3.077161e+09,3.101864e+09,3.120823e+09,3.154093e+09,2.740559e+09
855193,Uzbekistan,UZB,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Uzbek sum,NaN,Europe & Central Asia,Lower middle income,General trade system,Economic Policy & Debt: Purchasing power parity,...,9.472140e+10,1.032463e+11,1.116093e+11,1.210961e+11,1.311470e+11,1.419011e+11,1.532532e+11,1.651955e+11,1.784111e+11,9.472140e+10
312773,Equatorial Guinea,GNQ,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,CFA franc,National accounts have been revised from 1980 ...,Sub-Saharan Africa,High income: nonOECD,General trade system,Economic Policy & Debt: Purchasing power parity,...,2.949976e+10,3.475046e+10,3.521728e+10,3.207443e+10,3.416694e+10,3.700719e+10,3.547761e+10,3.562487e+10,3.240852e+10,2.949976e+10
96538,Albania,ALB,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Albanian lek,NaN,Europe & Central Asia,Upper middle income,General trade system,Economic Policy & Debt: Purchasing power parity,...,2.600106e+10,2.697892e+10,2.788342e+10,2.891791e+10,2.965536e+10,3.007646e+10,3.037745e+10,3.091646e+10,3.160484e+10,2.600106e+10
199158,Bulgaria,BGR,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Bulgarian lev,April 2011 database update: The National Stati...,Europe & Central Asia,Upper middle income,Special trade system,Economic Policy & Debt: Purchasing power parity,...,1.116393e+11,1.157005e+11,1.115513e+11,1.130283e+11,1.151928e+11,1.152284e+11,1.162218e+11,1.177662e+11,1.220263e+11,1.115513e+11
800218,Timor-Leste,TLS,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,U.S. dollar,"Based on official government statistics, natio...",East Asia & Pacific,Lower middle income,Special trade system,Economic Policy & Debt: Purchasing power parity,...,1.329418e+09,1.517851e+09,1.715186e+09,1.888782e+09,2.038638e+09,2.141015e+09,2.197397e+09,2.289388e+09,2.382863e+09,1.329418e+09
671943,Qatar,QAT,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Qatari riyal,NaN,Middle East & North Africa,High income: nonOECD,Special trade system,Economic Policy & Debt: Purchasing power parity,...,1.413661e+11,1.663363e+11,1.862244e+11,2.227101e+11,2.524980e+11,2.643333e+11,2.759911e+11,2.869724e+11,2.971629e+11,1.538512e+11
598643,Nepal,NPL,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,Nepalese rupee,Fiscal year end: July 14; reporting period for...,South Asia,Low income,General trade system,Economic Policy & Debt: Purchasing power parity,...,4.616292e+10,4.898100e+10,5.120135e+10,5.366742e+10,5.550383e+10,5.815757e+10,6.055883e+10,6.418568e+10,6.631851e+10,4.616292e+10
294448,Dominica,DMA,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,East Caribbean dollar,April 2012 database update: Based on official ...,Latin America & Caribbean,Upper middle income,Special trade system,Economic Policy & Debt: Purchasing power parity,...,6.835867e+08,7.322632e+08,7.236975e+08,7.285647e+08,7.268816e+08,7.190785e+08,7.147091e+08,7.463542e+08,7.273689e+08,6.835867e+08
23238,European Union,EUU,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,NaN,European Union aggregate.,NaN,NaN,NaN,Economic Policy & Debt: Purchasing power parity,...,1.747800e+13,1.760372e+13,1.683327e+13,1.717821e+13,1.748166e+13,1.740898e+13,1.742229e+13,NaN,NaN,1.683327e+13


In [ ]:
df_indicator_data

,Country Name,Country Code,Indicator Name,Indicator Code,Currency Unit,Special Notes,Region,Income Group,System of trade,Topic,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
1224,Arab World,ARB,Expenditure on education as % of total governm...,SE.XPD.TOTL.GB.ZS,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Expenditures,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1246,Arab World,ARB,"GDP per capita, PPP (constant 2011 internation...",NY.GDP.PCAP.PP.KD,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Economic Policy & Debt: Purchasing power parity,...,1.343403e+04,1.386042e+04,1.437783e+04,1.435481e+04,1.475905e+04,1.482591e+04,1.534280e+04,1.545497e+04,NaN,NaN
1248,Arab World,ARB,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Economic Policy & Debt: Purchasing power parity,...,4.260279e+12,4.500388e+12,4.779549e+12,4.882776e+12,5.132269e+12,5.265230e+12,5.561251e+12,5.714653e+12,NaN,NaN
1260,Arab World,ARB,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Expenditures,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1324,Arab World,ARB,"Gross enrolment ratio, primary and secondary, ...",UIS.GER.123.F,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,NaN,Secondary,...,7.694061e+01,7.665801e+01,7.747315e+01,7.841426e+01,7.945959e+01,8.052322e+01,8.216338e+01,8.091618e+01,81.176811,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884921,Zimbabwe,ZWE,"Net enrolment rate, secondary, gender parity i...",UIS.NER.23.GPI,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Secondary,...,NaN,NaN,NaN,NaN,NaN,NaN,1.002280e+00,1.011080e+00,NaN,NaN
885601,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.F,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Population,...,1.784010e+05,1.803770e+05,1.832270e+05,1.868440e+05,1.910540e+05,1.957320e+05,2.015990e+05,2.082110e+05,215118.000000,221749.0
885602,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.M,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Population,...,1.793050e+05,1.813000e+05,1.841870e+05,1.878030e+05,1.919210e+05,1.965500e+05,2.024060e+05,2.090290e+05,215964.000000,222623.0
885801,Zimbabwe,ZWE,"Primary completion rate, female (%)",SE.PRM.CMPT.FE.ZS,U.S. dollar,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,General trade system,Primary,...,NaN,NaN,NaN,NaN,NaN,NaN,9.035918e+01,9.093144e+01,NaN,NaN


In [ ]:
df_countries_data.copy()

df_countries_data

### 2. Feature Engineering


### 3. Modeling and Training


### 4. Evaluation
